# Omnivorous DINO Inference Demo

This notebook demonstrates how to load the open-sourced checkpoints of the **Omnivorous DINOv2** model, preprocess RGB and Depth inputs (including depth colorization), and run inference to extract aligned dense representations.

### Setup
Please install `flax`, `jax`, and `safetensors` if they are not already installed. Then import the `representations4d` package below.


In [ ]:
# Install dependencies if needed (uncomment if running in clean colab)
# !pip install flax jax safetensors pillow matplotlib numpy ml-collections

import os
import urllib.request
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image
from safetensors.flax import load_file
from flax.traverse_util import unflatten_dict

# Import from the ported package
from representations4d.omnivorous import vit

print("JAX devices:", jax.devices())


In [ ]:
# Define ViT-B/14 configuration matching the training setup
import ml_collections

vit_config = ml_collections.ConfigDict({
    'tokenize': True,
    'num_layers': 12,
    'hidden_size': 768,
    'mlp_dim': 3072,
    'num_heads': 12,
    'patches': ml_collections.ConfigDict({'size': (14, 14)}),
    'posembs': (37, 37),
    'pos_interpolation_method': 'bicubic',
    'pooling': 'tok',
})

# Instantiate models
student_model = vit.ViTBackbone(**vit_config)
teacher_model = vit.ViTBackbone(**vit_config)

# GCS Bucket details (crocsynth project)
BUCKET_URL = "https://storage.googleapis.com/omnivorous"
student_ckpt_name = "omnivorous_dinov2-vit_b.safetensors"
teacher_ckpt_name = "frozen_dinov2-vit_b.safetensors"

def download_and_load(ckpt_name, model):
  local_path = ckpt_name
  if not os.path.exists(local_path):
    url = f"{BUCKET_URL}/{ckpt_name}"
    print(f"Attempting to download {ckpt_name} from {url}...")
    try:
      urllib.request.urlretrieve(url, local_path)
      print("Downloaded successfully.")
    except Exception as e:
      print(f"Could not download from public URL (it might not be public yet): {e}")
      # Fallback to local tmp copy (useful for internal testing before bucket is public)
      original_tmp_name = "student_encoder.safetensors" if "student" in ckpt_name or "omnivorous" in ckpt_name else "teacher_encoder.safetensors"
      local_tmp = f"/tmp/omnivorous_export/{original_tmp_name}"
      if os.path.exists(local_tmp):
         print(f"Found local tmp copy at {local_tmp}. Using it.")
         local_path = local_tmp
      else:
         # Try also the renamed version in tmp
         local_tmp_renamed = f"/tmp/omnivorous_export/{ckpt_name}"
         if os.path.exists(local_tmp_renamed):
            print(f"Found renamed local tmp copy at {local_tmp_renamed}. Using it.")
            local_path = local_tmp_renamed
         else:
            raise FileNotFoundError(f"Could not find checkpoint {ckpt_name} in GCS or local tmp.")

  print(f"Loading weights from {local_path}...")
  state_dict = load_file(local_path)

  # Unflatten the flat safetensors dict to match Flax nested structure
  nested_params = unflatten_dict({tuple(k.split('.')): v for k, v in state_dict.items()})

  # Wrap in 'params' dict as expected by Flax init
  variables = {'params': nested_params}

  # Initialize model with dummy input to verify structure (optional but good practice)
  rng = jax.random.PRNGKey(0)
  dummy_input = jnp.zeros((1, 224, 224, 3))
  model_vars = model.init(rng, dummy_input, train=False)

  # Verify keys match
  expected_keys = set(model_vars['params'].keys())
  loaded_keys = set(variables['params'].keys())
  missing_keys = expected_keys - loaded_keys
  extra_keys = loaded_keys - expected_keys

  if missing_keys:
    print(f"Warning: Missing keys in loaded checkpoint: {missing_keys}")
  if extra_keys:
    print(f"Warning: Extra keys in loaded checkpoint: {extra_keys}")

  if not missing_keys and not extra_keys:
    print("Checkpoint keys match model structure perfectly!")

  return variables

try:
  print("--- Loading Student Model ---")
  student_vars = download_and_load(student_ckpt_name, student_model)
  print("\n--- Loading Teacher Model ---")
  teacher_vars = download_and_load(teacher_ckpt_name, teacher_model)
except Exception as e:
  print("\nError loading models:", e)
  print("Please ensure you have copied the checkpoints to GCS or they are present in /tmp/omnivorous_export/")


In [ ]:
# Inference, PCA Visualization, and Alignment Evaluation Helpers

import numpy as np
import matplotlib.pyplot as plt

def l2_normalize(x, axis=-1):
  """Helper to L2 normalize features along the channel dimension."""
  return x / jnp.maximum(jnp.linalg.norm(x, axis=axis, keepdims=True), 1e-8)


def get_features_and_score(model, variables, rgb_input, aux_input):
  """Runs inference and computes the average corresponding patch similarity."""
  outputs_rgb = model.apply(variables, rgb_input, train=False)
  outputs_aux = model.apply(variables, aux_input, train=False)

  # Extract dense tokens (excluding CLS at index 0)
  # final_tokens shape: [B, 257, 768]
  dense_rgb = outputs_rgb['final_tokens'][:, 1:, :]  # [1, 256, 768]
  dense_aux = outputs_aux['final_tokens'][:, 1:, :]  # [1, 256, 768]

  # Normalize
  dense_rgb_norm = l2_normalize(dense_rgb)[0]  # [256, 768]
  dense_aux_norm = l2_normalize(dense_aux)[0]  # [256, 768]

  # Compute corresponding patchwise cosine similarity
  patchwise_sims = jnp.sum(dense_rgb_norm * dense_aux_norm, axis=-1)  # [256]
  avg_score = jnp.mean(patchwise_sims)

  return avg_score, dense_rgb, dense_aux


def visualize_pca(features):
  """Applies PCA (via SVD) independently per image and returns a 16x16 RGB visualization."""
  # features shape: [1, 256, 768]
  feat = np.array(features[0])  # [256, 768]
  N, D = feat.shape

  # Center the features
  feat_centered = feat - np.mean(feat, axis=0, keepdims=True)

  # Run SVD (fast on CPU for 256x768)
  U, S, Vt = np.linalg.svd(feat_centered, full_matrices=False)

  # Project to top 3 components
  proj = U[:, :3] * S[:3]  # [256, 3]

  # Normalize to [0, 1] for RGB visualization
  proj_min = proj.min(axis=0, keepdims=True)
  proj_max = proj.max(axis=0, keepdims=True)
  proj_norm = (proj - proj_min) / (proj_max - proj_min + 1e-8)

  # Reshaping to 16x16 grid (for 14x14 patches on 224x224 image)
  img_pca = proj_norm.reshape(16, 16, 3)
  return img_pca


def run_demo(rgb_name, aux_name, title):
  """Runs the full evaluation and visualization pipeline for a pair of images."""
  print(f"\n==================================================")
  print(f"  Running Demo: {title}")
  print(f"==================================================")

  assets_dir = '../assets/'
  rgb_path = os.path.join(assets_dir, rgb_name)
  aux_path = os.path.join(assets_dir, aux_name)

  if not os.path.exists(rgb_path) or not os.path.exists(aux_path):
    print(f"Error: Assets not found in {assets_dir}. Please verify files exist.")
    return

  rgb_img = Image.open(rgb_path)
  aux_img = Image.open(aux_path)

  # Convert to numpy and normalize to [0, 1]
  rgb_np = np.array(rgb_img).astype(np.float32) / 255.0
  aux_np = np.array(aux_img).astype(np.float32) / 255.0

  # Resize helper
  def resize_image(img_np, size=(224, 224)):
    img = Image.fromarray((img_np * 255).astype(np.uint8))
    return np.array(img.resize(size)).astype(np.float32) / 255.0

  rgb_resized = resize_image(rgb_np)
  aux_resized = resize_image(aux_np)

  # Normalize using ImageNet mean/std (expected by ViT Backbone)
  IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
  IMAGENET_STD = np.array([0.229, 0.224, 0.225])

  rgb_input = (rgb_resized - IMAGENET_MEAN) / IMAGENET_STD
  aux_input = (aux_resized - IMAGENET_MEAN) / IMAGENET_STD

  rgb_input = np.expand_dims(rgb_input, axis=0)
  aux_input = np.expand_dims(aux_input, axis=0)

  # 1. Evaluate Frozen Teacher
  teacher_score, teacher_rgb_feat, teacher_aux_feat = get_features_and_score(
      teacher_model, teacher_vars, rgb_input, aux_input
  )

  # 2. Evaluate Trained Student
  student_score, student_rgb_feat, student_aux_feat = get_features_and_score(
      student_model, student_vars, rgb_input, aux_input
  )

  # 3. Compute PCA Visualizations (fit independently per image)
  teacher_rgb_pca = visualize_pca(teacher_rgb_feat)
  teacher_aux_pca = visualize_pca(teacher_aux_feat)

  student_rgb_pca = visualize_pca(student_rgb_feat)
  student_aux_pca = visualize_pca(student_aux_feat)

  # Plotting 2 rows, 3 columns figure
  fig, axs = plt.subplots(2, 3, figsize=(15, 10))

  # Column 1: Original Inputs (Resized)
  axs[0, 0].imshow(rgb_resized)
  axs[0, 0].set_title("RGB Input")
  axs[0, 0].axis('off')

  # Show the original auxiliary image (resized for display consistency)
  axs[1, 0].imshow(aux_resized)
  axs[1, 0].set_title(f"Auxiliary Input\n({aux_name})")
  axs[1, 0].axis('off')

  # Column 2: Frozen DINOv2 features (PCA)
  axs[0, 1].imshow(teacher_rgb_pca)
  axs[0, 1].set_title(f"Frozen DINOv2 RGB\n(Sim: {teacher_score:.4f})")
  axs[0, 1].axis('off')

  axs[1, 1].imshow(teacher_aux_pca)
  axs[1, 1].set_title(f"Frozen DINOv2 Aux\n(Sim: {teacher_score:.4f})")
  axs[1, 1].axis('off')

  # Column 3: Omnivorous DINOv2 features (PCA)
  axs[0, 2].imshow(student_rgb_pca)
  axs[0, 2].set_title(f"Omnivorous DINOv2 RGB\n(Sim: {student_score:.4f})")
  axs[0, 2].axis('off')

  axs[1, 2].imshow(student_aux_pca)
  axs[1, 2].set_title(f"Omnivorous DINOv2 Aux\n(Sim: {student_score:.4f})")
  axs[1, 2].axis('off')

  plt.tight_layout()
  plt.show()


In [ ]:
# Execute Demos for Hypersim and Artwork

# 1. Hypersim: RGB vs Depth
# Assumes the depth asset is already preprocessed/colorized as required by the model
run_demo(
    rgb_name='omnivorous-hypersim_rgb.png',
    aux_name='omnivorous-hypersim_depth.png',
    title='Hypersim (RGB vs Depth)'
)

# 2. Artwork: RGB vs X-ray
# Assumes the X-ray asset is already preprocessed/colorized as required by the model
run_demo(
    rgb_name='omnivorous-art_rgb.jpg',
    aux_name='omnivorous-art_xray.jpg',
    title='Artwork (RGB vs X-ray)'
)
